# Context-Based Idiom Meaning Identifier Using NLP
### B.Tech Computer Engineering NLP Mini-Project

This notebook demonstrates the end-to-end NLP workflow for **Context-Based Idiom Meaning Identification**:
1. Dataset Loading & Exploration
2. Class Distribution Analysis
3. Text Preprocessing & Lemmatization
4. Baseline Feature Extraction (TF-IDF)
5. Baseline Model Training (Logistic Regression)
6. Baseline Evaluation
7. Contextual Feature Extraction (Sentence Transformer Embeddings)
8. Contextual Model Training
9. Contextual Model Evaluation
10. Model Comparison & Metrics Summary
11. Interactive Prediction Demo

In [ ]:
# 1. Environment and Imports
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Add src directory to system path
sys.path.append(os.path.abspath('../src'))

from dataset_loader import load_dataset, get_train_test_split
from preprocessing import clean_text, normalize_for_tokens, lemmatize_text, detect_candidate_idiom
from idiom_dictionary import IDIOM_DICTIONARY, get_idiom_info
from predict import analyze_sentence

print('All modules imported successfully.')

## 2. Dataset Loading & Exploration

In [ ]:
# Load the benchmark dataset
df = load_dataset('../dataset/idioms.csv')
print(f'Total Dataset Shape: {df.shape}')
df.head(10)

## 3. Class and Idiom Distribution Analysis

In [ ]:
# Class distribution
print('Usage Type Distribution:')
print(df['usage_type'].value_counts())

plt.figure(figsize=(7, 3.5))
sns.countplot(data=df, x='usage_type', palette=['#10B981', '#F59E0B', '#6B7280'])
plt.title('Distribution of Usage Types in Dataset', fontweight='bold')
plt.xlabel('Usage Type')
plt.ylabel('Count')
plt.show()

## 4. Stratified Dataset Splitting

In [ ]:
# Perform 80-20 stratified split
X_train, X_test, y_train, y_test, df_train, df_test = get_train_test_split(csv_path='../dataset/idioms.csv')
print(f'Training samples: {len(X_train)} (80%)')
print(f'Testing samples:  {len(X_test)} (20%)')

print('\nTrain Class Distribution:\n', y_train.value_counts())
print('\nTest Class Distribution:\n', y_test.value_counts())

## 5. Text Preprocessing & Feature Extraction (TF-IDF Baseline)

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, accuracy_score

# Normalize text
X_train_clean = [normalize_for_tokens(s) for s in X_train]
X_test_clean = [normalize_for_tokens(s) for s in X_test]

# Encode labels
le = LabelEncoder()
y_train_enc = le.fit_transform(y_train)
y_test_enc = le.transform(y_test)

# TF-IDF with unigrams, bigrams, trigrams
tfidf = TfidfVectorizer(ngram_range=(1, 3), sublinear_tf=True, norm='l2')
X_train_tfidf = tfidf.fit_transform(X_train_clean)
X_test_tfidf = tfidf.transform(X_test_clean)

print(f'TF-IDF Feature Space Dimensions: {X_train_tfidf.shape[1]}')

## 6. Train Baseline Model (TF-IDF + Logistic Regression)

In [ ]:
baseline_model = LogisticRegression(C=2.0, max_iter=1000, class_weight='balanced', random_state=42)
baseline_model.fit(X_train_tfidf, y_train_enc)

y_pred_base = baseline_model.predict(X_test_tfidf)
acc_base = accuracy_score(y_test_enc, y_pred_base)
print(f'Baseline Test Accuracy: {acc_base*100:.2f}%\n')
print(classification_report(y_test_enc, y_pred_base, target_names=le.classes_, zero_division=0))

## 7. Contextual Feature Extraction (Sentence Transformers)

In [ ]:
from sentence_transformers import SentenceTransformer

# Load lightweight pretrained transformer
transformer_model = SentenceTransformer('all-MiniLM-L6-v2')

print('Generating dense 384-dimensional sentence embeddings...')
X_train_emb = transformer_model.encode(list(X_train), batch_size=32, normalize_embeddings=True)
X_test_emb = transformer_model.encode(list(X_test), batch_size=32, normalize_embeddings=True)

print(f'Train Embeddings Shape: {X_train_emb.shape}')
print(f'Test Embeddings Shape:  {X_test_emb.shape}')

## 8. Train Contextual Model (Sentence Transformer + Classifier)

In [ ]:
contextual_model = LogisticRegression(C=3.0, max_iter=1000, class_weight='balanced', random_state=42)
contextual_model.fit(X_train_emb, y_train_enc)

y_pred_ctx = contextual_model.predict(X_test_emb)
acc_ctx = accuracy_score(y_test_enc, y_pred_ctx)
print(f'Contextual Model Test Accuracy: {acc_ctx*100:.2f}%\n')
print(classification_report(y_test_enc, y_pred_ctx, target_names=le.classes_, zero_division=0))

## 9. Model Comparison & Confusion Matrix Analysis

In [ ]:
from sklearn.metrics import confusion_matrix, precision_score, recall_score, f1_score

metrics = [
    {
        'Model': 'TF-IDF + Logistic Regression (Baseline)',
        'Accuracy': accuracy_score(y_test_enc, y_pred_base),
        'Precision': precision_score(y_test_enc, y_pred_base, average='weighted', zero_division=0),
        'Recall': recall_score(y_test_enc, y_pred_base, average='weighted', zero_division=0),
        'F1-Score': f1_score(y_test_enc, y_pred_base, average='weighted', zero_division=0)
    },
    {
        'Model': 'Sentence Transformer + LogReg (Contextual)',
        'Accuracy': accuracy_score(y_test_enc, y_pred_ctx),
        'Precision': precision_score(y_test_enc, y_pred_ctx, average='weighted', zero_division=0),
        'Recall': recall_score(y_test_enc, y_pred_ctx, average='weighted', zero_division=0),
        'F1-Score': f1_score(y_test_enc, y_pred_ctx, average='weighted', zero_division=0)
    }
]
df_comparison = pd.DataFrame(metrics)
display(df_comparison.style.format({'Accuracy': '{:.2%}', 'Precision': '{:.2%}', 'Recall': '{:.2%}', 'F1-Score': '{:.2%}'}))

# Plot side by side confusion matrices
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5))
sns.heatmap(confusion_matrix(y_test_enc, y_pred_base), annot=True, fmt='d', cmap='Blues',
            xticklabels=le.classes_, yticklabels=le.classes_, ax=ax1, cbar=False)
ax1.set_title('Baseline Model Confusion Matrix', fontweight='bold')
ax1.set_xlabel('Predicted')
ax1.set_ylabel('True')

sns.heatmap(confusion_matrix(y_test_enc, y_pred_ctx), annot=True, fmt='d', cmap='Greens',
            xticklabels=le.classes_, yticklabels=le.classes_, ax=ax2, cbar=False)
ax2.set_title('Contextual Model Confusion Matrix', fontweight='bold')
ax2.set_xlabel('Predicted')
ax2.set_ylabel('True')
plt.tight_layout()
plt.show()

## 10. Sample Sentence Predictions & Context Disambiguation

In [ ]:
sample_sentences = [
    'After studying all night, Rahul was feeling under the weather.',
    'The child was sitting under the weather shelter.',
    'The weather was terrible yesterday.',
    'After the presentation, the teacher said I had hit the nail on the head.',
    'The carpenter swung the hammer to hit the nail on the head.',
    'The mathematics test was a piece of cake.'
]

for s in sample_sentences:
    res = analyze_sentence(s, model_type='contextual', models_dir='../models')
    print('Sentence:', s)
    print('  -> Expression: ', res.get('detected_expression'))
    print('  -> Usage:      ', res.get('usage'))
    print('  -> Confidence: ', f"{res.get('confidence')}%")
    print('  -> Meaning:    ', res.get('meaning'))
    print('  -> Explanation:', res.get('explanation'))
    print('-' * 85)